## Imports and setup
Clean imports for modelling, metrics, and SHAP.


In [ ]:
# from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
# from rdkit import DataStructs
# from rdkit.Chem import AllChem as Chem
# from rdkit.Chem import Draw
# from rdkit.Chem import Descriptors
# from rdkit.ML.Descriptors import MoleculeDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import random
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import LeaveOneOut
from sklearn import preprocessing
#from genetic_selection import GeneticSelectionCV
# from mordred import Calculator, descriptors
import pickle 
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## Load data
Read engineered matrices and targets used downstream.


In [ ]:
import os
os.getcwd()

In [ ]:
data = pd.read_csv("../data/allData.csv")

with open('../code/streamlit/features_a2.pkl', 'rb') as file:
    features_a = pickle.load(file)
with open('../code/streamlit/features_b2.pkl', 'rb') as file:
    features_b = pickle.load(file)

In [ ]:
data = data.loc[:, (data != 0).any(axis=0)]

df_no_zeroB = data[data['b'] != 0]
df_no_zeroB = df_no_zeroB.reset_index()
df_no_zeroB= df_no_zeroB[df_no_zeroB.columns[1:]]

In [ ]:
data.head(4)

In [ ]:
df_no_zeroB.head(4)


## Fit Extra Trees with custom validation
Train Extra Trees regressors using leave-one-out style evaluation and a custom train/test split.


In [ ]:
def extra_trees_loo_analysis_castom(X, y, test_size=5, random_state=42, picture = False, show_numb = False):
    """
    Полный анализ Extra Trees с Leave-One-Out и предсказаниями на новых данных
    """
    print("=" * 80)
    print("FULL EXTRA TREES ANALYSIS WITH LEAVE-ONE-OUT")
    print("=" * 80)
    
    X_train, X_new, y_train, y_new = train_test_split(
        X, y, test_size=test_size, random_state=random_state, shuffle=False
    )
    
    print(f"Размер тренировочных данных: {X_train.shape}")
    print(f"Размер новых данных (модель никогда не видела): {X_new.shape}")

    et_model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.6,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    
    print("\n" + "=" * 50)
    print("LEAVE-ONE-OUT CROSS-VALIDATION")
    print("=" * 50)
    
    loo = LeaveOneOut()
    loo_scores = []
    loo_predictions = []
    loo_true_values = []
    
    for train_idx, test_idx in loo.split(X_train):
        X_train_fold, X_test_fold = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_train_fold, y_test_fold = y_train.iloc[train_idx], y_train.iloc[test_idx]
        
        et_model.fit(X_train_fold, y_train_fold)
        y_pred_fold = et_model.predict(X_test_fold)
        
        mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
        loo_scores.append(-mse_fold)
        loo_predictions.extend(y_pred_fold)
        loo_true_values.extend(y_test_fold)
    
    print("\n" + "=" * 50)
    print("FINAL MODEL FIT")
    print("=" * 50)
    
    et_model.fit(X_train, y_train)
    
    y_new_pred = et_model.predict(X_new)
    original_y_new_pred = y_new_pred.copy()


    et_model_all = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.6,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    X_all = pd.concat([X_train, X_new])
    y_all = pd.concat([y_train, y_new])

    loo_all = LeaveOneOut()
    loo_scores_all = []
    loo_predictions_all = []
    loo_true_values_all = []

    for train_idx, test_idx in loo_all.split(X_all):
        X_train_fold, X_test_fold = X_all.iloc[train_idx], X_all.iloc[test_idx]
        y_train_fold, y_test_fold = y_all.iloc[train_idx], y_all.iloc[test_idx]
        
        et_model_all.fit(X_train_fold, y_train_fold)
        y_pred_fold = et_model_all.predict(X_test_fold)
        
        mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
        loo_scores_all.append(-mse_fold)
        loo_predictions_all.extend(y_pred_fold)
        loo_true_values_all.extend(y_test_fold)

    from sklearn.base import clone

    et_model_loo = clone(et_model_all)
    et_model_all.fit(X_all, y_all)

    y_train_pred = et_model_all.predict(X_all)

    train_r2 = r2_score(y_all, y_train_pred)
    train_mse = mean_squared_error(y_all, y_train_pred)
    train_mae = mean_absolute_error(y_all, y_train_pred)
    
    loo_r2 = r2_score(loo_true_values_all, loo_predictions_all)
    loo_mse = mean_squared_error(loo_true_values_all, loo_predictions_all)
    loo_mae = mean_absolute_error(loo_true_values_all, loo_predictions_all)
    
    new_r2 = r2_score(y_new, original_y_new_pred)
    new_mse = mean_squared_error(y_new, original_y_new_pred)
    new_mae = mean_absolute_error(y_new, original_y_new_pred)
    
    print("\n📊 RESULTS:")
    print("Metrics on training data:")
    print(f"  R²: {train_r2:.4f}, MSE: {train_mse:.4f}, MAE: {train_mae:.4f}")
    
    print("\nLeave-one-out metrics:")
    print(f"  R²: {loo_r2:.4f}, MSE: {loo_mse:.4f}, MAE: {loo_mae:.4f}")
    print(f"  LOO Score (mean neg_MSE): {np.mean(loo_scores_all):.4f} ± {np.std(loo_scores_all):.4f}")
    
    print("\nMetrics on NEW data (held out from training):")
    print(f"  R²: {new_r2:.4f}, MSE: {new_mse:.4f}, MAE: {new_mae:.4f}")
    
    if picture == True:
        plot_comprehensive_analysis(
            y_all, y_train_pred, 
            loo_true_values_all, loo_predictions_all,
            y_new, original_y_new_pred,
            X_new.index, show_numb
        )
        
        print_new_predictions_details(X_new, y_new, y_new_pred, X_new.index)
    return {
            'model': et_model_all,
            'LOOCV model': et_model_loo,
            'X_train': X_all,
            'y_train': y_all,
            'X_new': X_new,
            'y_new': y_new,
            'y_new_pred': y_new_pred,
            'loo_scores': loo_scores_all,
            'loo_predictions': loo_predictions_all,
            'loo_true_values': loo_true_values_all,
            'AE_LOOCV': np.abs((np.array(loo_true_values_all) - np.array(loo_predictions_all))),
            'AE_last': np.abs((np.array(y_all) - np.array(y_train_pred))),
            'metrics': {
                'train': {'r2': train_r2, 'mse': train_mse, 'mae': train_mae},
                'loo': {'r2': loo_r2, 'mse': loo_mse, 'mae': loo_mae},
                'new': {'r2': new_r2, 'mse': new_mse, 'mae': new_mae}
            }
    }
        
def plot_comprehensive_analysis(y_train, y_train_pred, loo_true, loo_pred, y_new, y_new_pred, new_indices, show_numb):
    """
    Комплексная визуализация результатов
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Comprehensive analysis of pretiction \n for "A" using features selected for "B"', fontsize=16, fontweight='bold')
    
    axes[0, 0].scatter(y_train, y_train_pred, alpha=0.6, color='blue', s=50)
    if show_numb == True:
        for i, idx in enumerate(y_train):
            axes[0, 0].annotate(f'{i}', (y_train[i], y_train_pred[i]), 
                            xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val = min(y_train.min(), y_train_pred.min())
    max_val = max(y_train.max(), y_train_pred.max())
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True Values')
    axes[0, 0].set_ylabel('Predicted Values')
    axes[0, 0].set_title('Parity Plot: predictions for training on all data after LOOCV')
    axes[0, 0].grid(True, alpha=0.3)
    r2_train = r2_score(y_train, y_train_pred)
    axes[0, 0].text(0.05, 0.95, f'R² = {r2_train:.3f}', transform=axes[0, 0].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    # 2. Parity Plot - LOO
    axes[0, 1].scatter(loo_true, loo_pred, alpha=0.6, color='green', s=50)
    if show_numb == True:
        for i, idx in enumerate(loo_true):
            axes[0, 1].annotate(f'{i}', (loo_true[i], loo_pred[i]), 
                            xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val_loo = min(min(loo_true), min(loo_pred))
    max_val_loo = max(max(loo_true), max(loo_pred))
    axes[0, 1].plot([min_val_loo, max_val_loo], [min_val_loo, max_val_loo], 'r--', linewidth=2)
    axes[0, 1].set_xlabel('True Values')
    axes[0, 1].set_ylabel('Predicted Values')
    axes[0, 1].set_title('Parity Plot: predictions for Leave-One-Out cross-validation (LOOCV)')
    axes[0, 1].grid(True, alpha=0.3)
    r2_loo = r2_score(loo_true, loo_pred)
    axes[0, 1].text(0.05, 0.95, f'R² = {r2_loo:.3f}', transform=axes[0, 1].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    axes[0, 2].scatter(y_new, y_new_pred, alpha=0.6, color='red', s=80, edgecolors='black')
    if show_numb == True:
        for i, idx in enumerate(new_indices):
            axes[0, 2].annotate(f'{idx}', (y_new.iloc[i], y_new_pred[i]), 
                            xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val_new = min(min(y_new), min(y_new_pred))
    max_val_new = max(max(y_new), max(y_new_pred))
    axes[0, 2].plot([min_val_new, max_val_new], [min_val_new, max_val_new], 'r--', linewidth=2)
    axes[0, 2].set_xlabel('True Values')
    axes[0, 2].set_ylabel('Predicted Values')
    axes[0, 2].set_title('Parity Plot: predictions for hold out sample')
    axes[0, 2].grid(True, alpha=0.3)
    r2_new = r2_score(y_new, y_new_pred)
    axes[0, 2].text(0.05, 0.95, f'R² = {r2_new:.3f}', transform=axes[0, 2].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_train = y_train - y_train_pred
    axes[1, 0].hist(residuals_train, bins=15, alpha=0.7, color='blue', edgecolor='black')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 0].set_xlabel('Residuals')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of prediction errors \n in training on all data after LOOCV')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].text(0.05, 0.95, f'Mean value: {residuals_train.mean():.3f}\nStandard deviation: {residuals_train.std():.3f}', 
                   transform=axes[1, 0].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_loo = np.array(loo_true) - np.array(loo_pred)
    axes[1, 1].hist(residuals_loo, bins=15, alpha=0.7, color='green', edgecolor='black')
    axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 1].set_xlabel('Residuals')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Residuals distribution errors for \n Leave-One-Out cross-validation (LOOCV)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].text(0.05, 0.95, f'Mean value: {residuals_loo.mean():.3f}\nStandard deviation: {residuals_loo.std():.3f}', 
                   transform=axes[1, 1].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_new = y_new - y_new_pred
    axes[1, 2].hist(residuals_new, bins=min(10, len(y_new)), alpha=0.7, color='red', edgecolor='black')
    axes[1, 2].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 2].set_xlabel('Residuals')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].set_title('Residuals Distribution errors for hold out sample')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].text(0.05, 0.95, f'Mean value: {residuals_new.mean():.3f}\nStandard deviation: {residuals_new.std():.3f}', 
                   transform=axes[1, 2].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def print_new_predictions_details(X_new, y_new, y_new_pred, indices):
    """
    Детальная информация о предсказаниях на новых данных
    """
    print("\n" + "=" * 80)
    print("DETAILED PREDICTION REPORT (NEW DATA)")
    print("=" * 80)
    
    predictions_df = pd.DataFrame({
        'Sample index': indices,
        'True value': y_new.values,
        'Predicted value': y_new_pred,
        'Error': y_new.values - y_new_pred,
        'Absolute error': np.abs(y_new.values - y_new_pred),
        'Relative error (%)': np.abs((y_new.values - y_new_pred) / y_new.values) * 100
    })
    
    print(predictions_df.round(4))
    
    print(f"\n📈 СТАТИСТИКА ОШИБОК НА НОВЫХ ДАННЫХ:")
    print(f"  Mean absolute error: {predictions_df['Absolute error'].mean():.4f}")
    print(f"  Max absolute error: {predictions_df['Absolute error'].max():.4f}")
    print(f"  Mean relative error: {predictions_df['Relative error (%)'].mean():.2f}%")
    print(f"  Std of errors: {predictions_df['Error'].std():.4f}")


## Model zoo
Define or wrap estimators compared in this notebook.


In [ ]:
print("Running Extra Trees diagnostics...")
results_b = extra_trees_loo_analysis_castom(df_no_zeroB[features_b[:25]], df_no_zeroB["b"], test_size=10, random_state=42, picture=True)

In [ ]:
print("Running Extra Trees diagnostics...")
results_a = extra_trees_loo_analysis_castom(data[features_a[:26]], data["a"], test_size=10, random_state=42, picture=True)

## Reversed-feature experiment
Use the feature subset selected for target **b** while fitting a model that predicts **a** (sanity / transfer check).


In [ ]:
print("Running Extra Trees diagnostics...")
resultsA_b = extra_trees_loo_analysis_castom(data[features_b[:25]], data["a"], test_size=10, random_state=42, picture=True)

## Save artefacts
Persist trained models and ordered feature-name lists for the Streamlit app.


In [ ]:
# import joblib
# joblib.dump(results_a['model'], 'streamlit/model_a2.pkl')
# joblib.dump(features_a[:26], 'streamlit/features_a2.pkl')

## SHAP analysis
Global and local explanations for the tree-based models.


In [ ]:
import shap

In [ ]:
def shap_analysis_extra_trees(model, X, y=None, data = None, sample_size=100, random_state=42, rows_to_annotate = None, threshold = None, picture =True):
    """
    SHAP analysis for Extra Trees regression
    """
    print("=" * 80)
    print("SHAP ANALYSIS FOR EXTRA TREES REGRESSION")
    print("=" * 80)
    
    if len(X) > sample_size:
        X_sample = X.sample(n=sample_size, random_state=random_state)
        print(f"Using sample of {sample_size} instances for SHAP analysis")
    else:
        X_sample = X
        print(f"Using all {len(X)} instances for SHAP analysis")
    
    explainer = shap.TreeExplainer(model)
    
    print("Calculating SHAP values...")
    shap_values = explainer.shap_values(X_sample)
    
    expected_value = explainer.expected_value
    if isinstance(expected_value, np.ndarray):
        expected_value = expected_value[0] if len(expected_value) > 0 else expected_value
    # print(f"Base value (expected value): {expected_value:.4f}")
    print(f"SHAP values shape: {shap_values.shape}")
    
    shap_explanation = shap.Explanation(
        values=shap_values,
        base_values=expected_value,
        data=X_sample.values,
        feature_names=X_sample.columns.tolist()
    )
    
    print("\n📈 Creating summary plot with index annotations...")
    plt.figure(figsize=(12, 8))
    
    shap.summary_plot(shap_values, X_sample, show=False)

    shap_df = _add_simple_index_annotations(shap_values, X_sample, df = data, threshold=threshold, rows_to_annotate= rows_to_annotate)
    
    plt.title(f"SHAP Summary Plot - Feature Importance", 
              fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    feature_importance_df = pd.DataFrame({
        'feature': X_sample.columns,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False)
    
    top_20_features = feature_importance_df.head(20)
    print("\n🔝 Top 20 most important features by mean |SHAP|:")
    print(top_20_features.to_string(index=False))
    
    if picture == True:
        print("📊 Creating feature importance plot...")
        plt.figure(figsize=(12, 6))
        shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
        plt.title("SHAP Feature Importance (Mean |SHAP|)", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # 3. Beeswarm plot
        print("🐝 Creating beeswarm plot...")
        plt.figure(figsize=(12, 8))
        shap.plots.beeswarm(shap_explanation, show=False)
        plt.title("SHAP Beeswarm Plot - Feature Effects", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("🌊 Creating waterfall plot for first instance...")
        plt.figure(figsize=(14, 8))
        shap.plots.waterfall(shap_explanation[0], show=False, max_display=15)
        plt.title(f"SHAP Waterfall Plot - Instance: {X_sample.index[0]}", 
                fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("⚡ Creating force plot...")
        plt.figure(figsize=(14, 4))
        shap.plots.force(shap_explanation[0], matplotlib=True, show=False)
        plt.title(f"SHAP Force Plot - Instance: {X_sample.index[0]}", 
                fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("📈 Creating decision plot...")
        plt.figure(figsize=(14, 10))
        shap.decision_plot(expected_value, shap_values[:10], 
                        X_sample.iloc[:10], 
                        feature_names=list(X_sample.columns),
                        show=False)
        plt.title("SHAP Decision Plot (First 10 Instances)", 
                fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # print("📊 Creating dependence plots for top 4 features...")
        # top_features = get_top_features_from_shap(shap_values, X_sample.columns, n=4)
        
        # fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        # axes = axes.ravel()
        
        # for i, feature in enumerate(top_features):
        #     shap.dependence_plot(feature, shap_values, X_sample, 
        #                        ax=axes[i], show=False)
        #     axes[i].set_title(f'SHAP Dependence: {feature}', fontweight='bold', fontsize=12)
        #     axes[i].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
    results = {
        'shap_df':shap_df,
        'explainer': explainer,
        'shap_values': shap_values,
        'expected_value': expected_value,
        'X_sample': X_sample,
        'shap_explanation': shap_explanation,
        'top_20': top_20_features
    }
    
    return results

def _add_simple_index_annotations(shap_values, X_sample, df = None, threshold=None, rows_to_annotate = None):
    
    ax = plt.gca()
    print(f"Трешхолды:{threshold}")
    y_tick_labels = [label.get_text() for label in ax.get_yticklabels()]
    feature_names = X_sample.columns.tolist()

    threshold_dict = dict(zip(rows_to_annotate, threshold))

    position_to_feature_idx = {}
    print (y_tick_labels)
    print (rows_to_annotate)
    for position, feature_name in enumerate(y_tick_labels):
        if feature_name in rows_to_annotate:
            print(feature_name)
            position_to_feature_idx[position] = feature_names.index(feature_name)
    
    outlier_data = []

    # print(position_to_feature_idx)
    
    for position, feature_idx in position_to_feature_idx.items():
        feature_name = feature_names[feature_idx]
        feature_shap = shap_values[:, feature_idx]
        high_impact_indices = np.where(np.abs(feature_shap) > threshold_dict[X_sample.columns[feature_idx]])[0]
        
        x_positions = []
        for idx in high_impact_indices:
            shap_val = feature_shap[idx]
            sample_index = X_sample.index[idx]
            feature_value = X_sample.iloc[idx, feature_idx]
            
            outlier_data.append({
                'feature': feature_name,
                'feature_value': feature_value,
                'index': sample_index,
                'reaction_name': df["inp+inp=out"][sample_index],
                'shap_value': shap_val,
                'abs_shap': abs(shap_val)
            })
            
            x_positions.append((shap_val, sample_index, position))
        
        x_positions.sort(key=lambda x: x[0])
        
        for i, (x, text, y_pos) in enumerate(x_positions):
            y_offset = 10 if i % 2 == 0 else -10
            x_offset = 5 + (i % 3) * 8
            
            ax.annotate(text,
                       xy=(x, y_pos),
                       xytext=(x_offset, y_offset),
                       textcoords='offset points',
                       fontsize=6,
                       alpha=0.7,
                       bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.5))
    
    # _print_outlier_table_pandas(outlier_data, threshold)
    
    return outlier_data

In [ ]:
rows_to_annotate = ["reagent2_GATS1p", "reagent1_ATSC7v", "reagent2_AATS1p", "reagent2_AATSC3p", "reagent2_TPSA", "reagent2_RPSA"]

In [ ]:
# aut_SHAP_b = None
shap_resultsB = shap_analysis_extra_trees(
    model=results_b["model"], 
    X = df_no_zeroB[features_b[:25]], 
    data = df_no_zeroB,
    y = df_no_zeroB["b"],
    rows_to_annotate = rows_to_annotate,
    threshold = [5, 4.5, 9, 9, 5, 5]#[2] * len(rows_to_annotate)
    )
df_shap_b = pd.DataFrame(shap_resultsB['shap_df'])

In [ ]:
b = shap_resultsB["top_20"]["feature"].to_list()

In [ ]:
b

In [ ]:
shap_resultsA = shap_analysis_extra_trees(
    model=results_a["model"], 
    X = data[features_a[:26]], 
    data = data,
    y = data["a"],
    rows_to_annotate = rows_to_annotate,
    threshold = [5, 4.5, 9, 9, 5, 5]#[2] * len(rows_to_annotate)
    )

In [ ]:
a = shap_resultsA["top_20"]["feature"].to_list()

In [ ]:
a

In [ ]:
aa = ["A_" + i for i in a]

In [ ]:
bb = ["B_" + i for i in b]

In [ ]:
len(aa+bb)

In [ ]:
m = aa+bb
o = a+b
for i in range(0, len(m)):
    data[m[i]] = data[o[i]]


In [ ]:
df = pd.concat([data["inp+inp=out"],  data[m]], axis=1)

In [ ]:
df

In [ ]:
data.head()

In [ ]:
df.to_csv("../data/ProdactionData3.csv")